# Session 1: Modern LLM Foundations for Agentic Systems (Instructor Notebook -- Full Solutions)

This is the **instructor version** of Session 1. It contains the same demos as the student notebook, plus **complete, working solutions** for all four tasks. Use this as your reference during class delivery.

## Learning Objectives

By the end of this session, students will be able to:

1. **Set up and configure** LLM API connections using the OpenAI Python SDK
2. **Understand tokenization** and how context windows affect LLM behavior
3. **Control model output** by tuning parameters such as temperature, top_p, and max_tokens
4. **Use system messages** to shape LLM response behavior and persona
5. **Build basic LLM pipelines** by chaining multiple API calls together
6. **Lay the groundwork** for conversational agents with message history management
7. **Generate and compare embeddings** to understand vector representations of text
8. **Visualize embeddings** using dimensionality reduction and inspect semantic similarity
9. **Understand how LLMs support reasoning** and agent behavior through emergent capabilities

In [1]:
from dotenv import load_dotenv
load_dotenv()  # Load environment variables from .env

# ============================================================
# Imports and Setup
# ============================================================

import openai
import tiktoken
import os
import json
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

# Ensure your OpenAI API key is set as an environment variable:
#   export OPENAI_API_KEY="sk-..."
# Or uncomment the line below and paste your key (not recommended for production):
# os.environ["OPENAI_API_KEY"] = "sk-..."

print("All imports successful!")
print(f"OpenAI SDK version: {openai.__version__}")

from dotenv import load_dotenv
load_dotenv()  # Load environment variables from .env

All imports successful!
OpenAI SDK version: 2.31.0


True

---
## Demos (Follow Along)

The following five demos are fully coded. Run each cell, observe the output, and make sure you understand what is happening before moving on.

### Demo 1: Setting Up LLM API Connections

In this demo we initialize the OpenAI client and make our first chat completion call. The `client` object handles authentication, request formatting, and response parsing for us.

**Scenario:** A McKinsey team needs to quickly query an LLM to summarize industry concepts for a client engagement kickoff.

In [2]:
# Demo 1 - Setting Up LLM API Connections

# Step 1: Initialize the OpenAI client
# The client automatically reads OPENAI_API_KEY from the environment.
client = openai.OpenAI()

# Step 2: Make a simple chat completion call
# Scenario: A McKinsey analyst needs a quick definition for a client deck
response = client.chat.completions.create(
    model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
    messages=[
        {"role": "user", "content": "What is a digital transformation strategy? Explain in two sentences for a McKinsey client presentation."}
    ],
    max_tokens=int(os.getenv("DEFAULT_MAX_TOKENS", "100"))
)

# Step 3: Print the response
print("Model used :", response.model)
print("Finish reason:", response.choices[0].finish_reason)
print("\nAssistant response:")
print(response.choices[0].message.content)

Model used : gpt-4o-mini-2024-07-18
Finish reason: stop

Assistant response:
A digital transformation strategy is a comprehensive plan that outlines how an organization will leverage digital technologies to improve processes, enhance customer experiences, and drive innovation across all areas of the business. By aligning technology investments with strategic goals, this approach enables companies to remain competitive and agile in a rapidly changing market landscape.


### Demo 2: Understanding Tokenization and Context Windows

Tokens are the fundamental units that LLMs process. A token can be a word, part of a word, or even a single character. Understanding tokenization helps you:
- Estimate API costs (billing is per-token)
- Stay within context window limits
- Write more efficient prompts

In [3]:
# Demo 2 - Understanding Tokenization and Context Windows

# Step 1: Get the tokenizer for gpt-4o-mini
encoder = tiktoken.encoding_for_model("gpt-4o-mini")

# Step 2: Encode various consulting-related texts and count tokens
texts = [
    "McKinsey & Company",
    "Digital transformation is reshaping how Fortune 500 companies create value across their organizations.",
    "The client's operating model requires restructuring to achieve the target $200M in synergies.",
    "Organizational restructuring and operational efficiency optimization",
    "def calculate_market_size(tam, penetration_rate):\n    sam = tam * penetration_rate\n    return sam * 0.15  # McKinsey target share"
]

print("Token counts for different consulting texts:")
print("=" * 60)
for text in texts:
    tokens = encoder.encode(text)
    print(f"\nText   : {text[:60]}{'...' if len(text) > 60 else ''}")
    print(f"Chars  : {len(text)}")
    print(f"Tokens : {len(tokens)}")
    print(f"Token IDs (first 10): {tokens[:10]}")

# Step 3: Demonstrate context window limits
print("\n" + "=" * 60)
print("Context window reference:")
print("  gpt-4o-mini : 128,000 tokens")
print("  gpt-4o      : 128,000 tokens")
print("  gpt-3.5-turbo: 16,385 tokens")

# Simulate a long due diligence report consuming the window
long_text = "The target company operates in a fragmented market with strong growth potential. " * 500
long_tokens = encoder.encode(long_text)
print(f"\nLong due diligence excerpt ({len(long_text)} chars) = {len(long_tokens)} tokens")
print(f"That is {len(long_tokens)/128000*100:.2f}% of the gpt-4o-mini context window.")

Token counts for different consulting texts:

Text   : McKinsey & Company
Chars  : 18
Tokens : 5
Token IDs (first 10): [41691, 73519, 28230, 827, 9709]

Text   : Digital transformation is reshaping how Fortune 500 companie...
Chars  : 102
Tokens : 16
Token IDs (first 10): [44057, 26242, 382, 132104, 19327, 1495, 57526, 220, 3234, 6005]

Text   : The client's operating model requires restructuring to achie...
Chars  : 93
Tokens : 17
Token IDs (first 10): [976, 122281, 14359, 2359, 10860, 116848, 316, 12998, 290, 3783]

Text   : Organizational restructuring and operational efficiency opti...
Chars  : 68
Tokens : 8
Token IDs (first 10): [19888, 482, 1953, 116848, 326, 33216, 17456, 34658]

Text   : def calculate_market_size(tam, penetration_rate):
    sam = ...
Chars  : 129
Tokens : 33
Token IDs (first 10): [1314, 17950, 165999, 5044, 2025, 313, 11, 86834, 20477, 1883]

Context window reference:
  gpt-4o-mini : 128,000 tokens
  gpt-4o      : 128,000 tokens
  gpt-3.5-turbo: 16,385 tokens



### Demo 3: Exploring Model Parameters

LLMs expose several parameters that control the randomness and length of generated text:

| Parameter | Description |
|-----------|-------------|
| `temperature` | Controls randomness. 0 = deterministic, 1 = creative |
| `top_p` | Nucleus sampling. Lower values = more focused output |
| `max_tokens` | Maximum number of tokens in the response |

In [4]:
# Demo 3 - Exploring Model Parameters

client = openai.OpenAI()
prompt = "Write a one-sentence executive summary of generative AI's impact on the management consulting industry."

# --- Temperature comparison ---
print("TEMPERATURE COMPARISON")
print("=" * 60)

for temp in [0.0, 0.5, 1.0]:
    response = client.chat.completions.create(
        model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
        messages=[{"role": "user", "content": prompt}],
        temperature=temp,
        max_tokens=int(os.getenv("DEFAULT_MAX_TOKENS", "80"))
    )
    print(f"\nTemperature {temp}:")
    print(f"  {response.choices[0].message.content}")

# --- top_p comparison ---
print("\n" + "=" * 60)
print("TOP_P COMPARISON")
print("=" * 60)

for top_p in [0.1, 0.5, 1.0]:
    response = client.chat.completions.create(
        model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
        messages=[{"role": "user", "content": prompt}],
        top_p=top_p,
        max_tokens=int(os.getenv("DEFAULT_MAX_TOKENS", "80"))
    )
    print(f"\ntop_p {top_p}:")
    print(f"  {response.choices[0].message.content}")

# --- max_tokens comparison ---
print("\n" + "=" * 60)
print("MAX_TOKENS COMPARISON")
print("=" * 60)

for max_tok in [10, 30, 100]:
    response = client.chat.completions.create(
        model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
        messages=[{"role": "user", "content": prompt}],
        temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0.5")),
        max_tokens=max_tok
    )
    text = response.choices[0].message.content
    print(f"\nmax_tokens={max_tok} (finish_reason={response.choices[0].finish_reason}):")
    print(f"  {text}")

TEMPERATURE COMPARISON

Temperature 0.0:
  Generative AI is revolutionizing the management consulting industry by enhancing data analysis, streamlining report generation, and enabling more personalized client solutions, ultimately driving efficiency and innovation in service delivery.

Temperature 0.5:
  Generative AI is transforming the management consulting industry by enhancing data analysis, streamlining report generation, and enabling more personalized client solutions, ultimately driving efficiency and innovation in service delivery.

Temperature 1.0:
  Generative AI is revolutionizing the management consulting industry by enhancing data analysis, streamlining decision-making processes, and enabling more personalized client solutions, ultimately driving greater efficiency and innovation in service delivery.

TOP_P COMPARISON

top_p 0.1:
  Generative AI is revolutionizing the management consulting industry by enhancing data analysis, streamlining report generation, and enabling mo

### Demo 4: Comparing LLM Response Behaviors

The **system message** is a powerful lever for controlling how the LLM behaves. By changing the system message, you can make the same model act as different McKinsey expert "personas," each with its own tone, expertise, and communication style — mirroring how different practice areas approach the same client question.

In [5]:
# Demo 4 - Comparing LLM Response Behaviors

client = openai.OpenAI()

user_question = "Our retail client is experiencing a 15% decline in same-store sales. What should we investigate?"

personas = {
    "McKinsey Partner (Strategy)": (
        "You are a McKinsey Senior Partner leading the Strategy & Corporate Finance practice. "
        "You think in terms of competitive dynamics, portfolio strategy, and value creation. "
        "Use structured, MECE frameworks and speak with authority."
    ),
    "McKinsey Associate (Operations)": (
        "You are a McKinsey Associate in the Operations practice. You focus on supply chain, "
        "store-level performance metrics, and operational efficiency. Be data-driven and suggest "
        "specific analyses to run."
    ),
    "McKinsey Expert (Digital & Analytics)": (
        "You are a McKinsey Digital & Analytics expert. You focus on digital channels, "
        "customer analytics, omnichannel strategy, and technology enablement. "
        "Recommend data-driven approaches and digital solutions."
    ),
}

print(f"Client Question: {user_question}")
print("=" * 60)

for persona_name, system_msg in personas.items():
    response = client.chat.completions.create(
        model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
        messages=[
            {"role": "system", "content": system_msg},
            {"role": "user", "content": user_question}
        ],
        temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0.7")),
        max_tokens=int(os.getenv("DEFAULT_MAX_TOKENS", "200"))
    )
    print(f"\n--- {persona_name} ---")
    print(response.choices[0].message.content)
    print()

Client Question: Our retail client is experiencing a 15% decline in same-store sales. What should we investigate?

--- McKinsey Partner (Strategy) ---
To address the 15% decline in same-store sales for your retail client, we should adopt a structured approach using a MECE (Mutually Exclusive, Collectively Exhaustive) framework. This will help us identify the root causes and potential solutions systematically. We can categorize our investigation into three main areas: Market Dynamics, Customer Insights, and Operational Efficiency.

### 1. Market Dynamics
   - **Competitive Landscape**: 
     - Analyze competitors’ performance and strategies. Are they gaining market share? What promotions or innovations are they implementing?
     - Assess any new entrants or disruptive business models affecting the market.
   - **Economic Factors**: 
     - Evaluate macroeconomic indicators (e.g., consumer confidence, unemployment rates) that may impact consumer spending.
     - Investigate regional eco

### Demo 5: Building a Basic LLM Pipeline

A pipeline chains multiple LLM calls together, where the output of one step becomes the input of the next. This pattern is fundamental to agentic systems and mirrors how McKinsey structures analysis — first synthesize raw research, then extract structured insights for client delivery.

In [6]:
# Demo 5 - Building a Basic LLM Pipeline

client = openai.OpenAI()

def call_llm(system_message, user_message, temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0.5")), max_tokens=int(os.getenv("DEFAULT_MAX_TOKENS", "300"))):
    """Helper function to make an LLM call with a system and user message."""
    response = client.chat.completions.create(
        model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
        messages=[
            {"role": "system", "content": system_message},
            {"role": "user", "content": user_message}
        ],
        temperature=temperature,
        max_tokens=max_tokens
    )
    return response.choices[0].message.content


# --- Pipeline: Summarize Industry Research -> Extract Key Insights ---

industry_research = """
The global management consulting market is undergoing a fundamental transformation driven by 
generative AI and advanced analytics. McKinsey's own research indicates that up to 70% of 
business activities could be automated with current technology. Traditional strategy engagements 
that once required weeks of analyst research can now be augmented with AI-driven market analysis 
in days. However, the human element remains critical — clients value the judgment, relationship 
management, and change leadership that experienced consultants provide. The shift is creating 
new service lines around AI implementation, responsible AI governance, and digital capability 
building. Firms that successfully integrate AI into their delivery model are seeing 30-40% 
improvements in engagement efficiency while maintaining or improving quality scores. The key 
challenges include managing data confidentiality across client engagements, ensuring AI outputs 
meet McKinsey's quality standards, building consultant capabilities in prompt engineering and 
AI-augmented analysis, and maintaining the trusted advisor relationship in an AI-augmented world.
"""

print("PIPELINE STEP 1: Summarize Industry Research")
print("=" * 60)

summary = call_llm(
    system_message="You are a McKinsey research analyst. Summarize the given text in 2-3 sentences suitable for a partner briefing.",
    user_message=f"Summarize this industry research:\n\n{industry_research}"
)
print(summary)

print("\nPIPELINE STEP 2: Extract Key Insights for Client Deck")
print("=" * 60)

key_points = call_llm(
    system_message="You are a McKinsey engagement manager preparing a client presentation. Extract exactly 5 key insights as a numbered list, each suitable for a slide bullet point.",
    user_message=f"Extract the key insights from this summary:\n\n{summary}"
)
print(key_points)

print("\nPIPELINE RESULT: Complete output")
print("=" * 60)
print(f"Original research : {len(industry_research.split())} words")
print(f"Summary length    : {len(summary.split())} words")
print(f"\nKey Insights for Client Deck:\n{key_points}")

PIPELINE STEP 1: Summarize Industry Research
The global management consulting market is rapidly evolving due to generative AI and advanced analytics, with McKinsey's research suggesting that up to 70% of business activities could be automated. While AI enhances efficiency—improving engagement by 30-40%—the human element remains essential for client relationships and change leadership. Key challenges include data confidentiality, maintaining quality standards, upskilling consultants in AI tools, and preserving the trusted advisor role amidst this technological shift.

PIPELINE STEP 2: Extract Key Insights for Client Deck
1. The global management consulting market is evolving rapidly, with generative AI and advanced analytics poised to automate up to 70% of business activities.

2. AI implementation can enhance efficiency, leading to a 30-40% improvement in client engagement.

3. Despite technological advancements, the human element is crucial for maintaining client relationships and eff

### Demo 6: Embeddings and Vector Representations

Embeddings convert text into dense numerical vectors that capture semantic meaning. Two sentences with similar meaning will have similar embedding vectors, even if they use different words. This is the foundation for:
- Semantic search across McKinsey knowledge bases and past engagement documents
- Clustering client industries and market research by topic
- Similarity-based retrieval for RAG-powered consulting assistants

**Instructor Note:** Emphasize that embeddings capture *meaning*, not just keywords. "Revenue growth decelerated in Q3" and "Sales momentum slowed in the third quarter" will have very similar vectors despite using different words.

### Task 1: Configure and Test Multiple LLM Provider Connections

**Approach:** The solution creates an OpenAI client, sends a consulting-relevant test message with constrained `max_tokens`, and returns the response content. We wrap the call in a try/except so students can see how to handle connection errors gracefully — critical for production consulting tools.

> ---
> **INSTRUCTOR GUIDANCE**
>
> **How to Conduct This Exercise:** Start by verifying everyone's API key is working before diving in. Give students 5-7 minutes. Walk the room and help anyone with environment issues first -- this is their first API call, so unblock quickly. After time is up, live-code the solution and emphasize the try/except pattern for production robustness.
>
> **Common Student Mistakes:**
> - Forgetting to set `OPENAI_API_KEY` as an environment variable (most common Day 1 blocker)
> - Hardcoding the API key directly in the notebook instead of using `os.environ`
> - Not handling exceptions -- students often skip try/except and get unreadable stack traces
> - Confusing `response` (the full API object) with `response.choices[0].message.content` (the actual text)
>
> **Skippable?** NO -- CRITICAL -- this is the first API call. Every subsequent task depends on students having a working connection. Do NOT skip.

In [7]:
# Task 1 - SOLUTION: Configure and Test Multiple LLM Provider Connections

def test_llm_connection():
    """Initialize OpenAI client and test with a consulting-relevant message."""
    # Step 1: Create the client. It reads OPENAI_API_KEY from the environment.
    client = openai.OpenAI()

    try:
        # Step 2: Send a McKinsey-relevant test message.
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "user", "content": "Define 'MECE' as used in McKinsey problem-solving. Respond in one sentence."}
            ],
            max_tokens=50
        )

        # Step 3: Extract and return the response text.
        result = response.choices[0].message.content
        print(f"Connection successful! Response: {result}")
        return result

    except openai.AuthenticationError:
        print("ERROR: Invalid API key. Check your OPENAI_API_KEY environment variable.")
        return None
    except openai.APIConnectionError:
        print("ERROR: Could not connect to the OpenAI API. Check your network.")
        return None
    except Exception as e:
        print(f"ERROR: Unexpected error -- {e}")
        return None


# Test the function
result = test_llm_connection()
print(f"\nReturned value: {result}")

Connection successful! Response: 'MECE' stands for 'Mutually Exclusive, Collectively Exhaustive,' a principle used in problem-solving to ensure that categories or options are distinct and cover all possible scenarios, preventing overlap and omissions in analysis.

Returned value: 'MECE' stands for 'Mutually Exclusive, Collectively Exhaustive,' a principle used in problem-solving to ensure that categories or options are distinct and cover all possible scenarios, preventing overlap and omissions in analysis.


### Task 2: Analyze Token Usage and Optimize Prompt Length

**Approach:** We use `tiktoken` to count tokens in a consulting research brief. If the count exceeds the limit, we split at sentence boundaries and rebuild. This is directly relevant to production workflows where long client documents must fit within context windows — a common challenge when feeding due diligence reports or market research into LLMs.

> ---
> **INSTRUCTOR GUIDANCE**
>
> **How to Conduct This Exercise:** Give students 8-10 minutes. This task has more logic than Task 1. Suggest students start by just counting tokens (Step 1-2) before implementing truncation. After the exercise, discuss edge cases: What if the prompt has no periods? What about very long single sentences? Use this as a bridge to explain why context window management matters for RAG on Day 3.
>
> **Common Student Mistakes:**
> - Using `len(prompt.split())` (word count) instead of `tiktoken` for token counting -- emphasize that tokens != words
> - Splitting on `'.'` instead of `'. '` which breaks abbreviations like 'U.S.' and decimal numbers like '$2.3B'
> - Not adding the period back after splitting -- the rebuilt text loses sentence endings
> - Forgetting to handle the case where the prompt is already under the limit (should return as-is)
> - Off-by-one errors in the running token count -- not accounting for spaces between reassembled sentences
>
> **Skippable?** YES -- CAN SKIP if running behind schedule. The core concept (tokens != words) can be conveyed verbally using Demo 2. Students will encounter tiktoken again in Day 3 RAG sessions. If skipping, at minimum show the solution and discuss the sentence-boundary truncation idea.

In [8]:
# Task 2 - SOLUTION: Analyze Token Usage and Optimize Prompt Length

def optimize_prompt(prompt, max_tokens=500):
    """Count tokens and truncate at sentence boundaries if over the limit.

    Returns a dict with:
        - original_tokens: token count of the input prompt
        - optimized_text: the (possibly truncated) prompt
        - optimized_tokens: token count of the optimized prompt
        - was_truncated: whether truncation was applied
    """
    # Step 1: Get the encoder for the target model.
    encoder = tiktoken.encoding_for_model("gpt-4o-mini")

    # Step 2: Count tokens in the original prompt.
    original_tokens = len(encoder.encode(prompt))

    # Step 3: If under the limit, return as-is.
    if original_tokens <= max_tokens:
        return {
            "original_tokens": original_tokens,
            "optimized_text": prompt,
            "optimized_tokens": original_tokens,
            "was_truncated": False
        }

    # Step 4: Split into sentences and rebuild until we hit the budget.
    sentences = prompt.split(". ")
    optimized_parts = []
    running_token_count = 0

    for sentence in sentences:
        # Add the period back (it was removed by split) except for the last fragment.
        candidate = sentence if sentence.endswith(".") else sentence + "."
        candidate_tokens = len(encoder.encode(candidate))

        if running_token_count + candidate_tokens > max_tokens:
            break  # Adding this sentence would exceed the limit.

        optimized_parts.append(candidate)
        running_token_count += candidate_tokens

    optimized_text = " ".join(optimized_parts)
    optimized_tokens = len(encoder.encode(optimized_text))

    return {
        "original_tokens": original_tokens,
        "optimized_text": optimized_text,
        "optimized_tokens": optimized_tokens,
        "was_truncated": True
    }


# Test with a long McKinsey-style market research brief
long_prompt = (
    "The global healthcare market is projected to reach $12 trillion by 2028. "
    "Digital health solutions are growing at 25% CAGR across developed markets. "
    "Regulatory tailwinds in the US and EU are accelerating telehealth adoption. "
    "McKinsey estimates that value-based care models could save $1 trillion annually. "
    "The competitive landscape is consolidating as major players pursue M&A strategies. "
) * 20  # Repeat to simulate a long research document

result = optimize_prompt(long_prompt, max_tokens=50)
print(f"Original tokens : {result['original_tokens']}")
print(f"Optimized tokens: {result['optimized_tokens']}")
print(f"Was truncated   : {result['was_truncated']}")
print(f"\nOptimized text:\n{result['optimized_text']}")

Original tokens : 1481
Optimized tokens: 44
Was truncated   : True

Optimized text:
The global healthcare market is projected to reach $12 trillion by 2028. Digital health solutions are growing at 25% CAGR across developed markets. Regulatory tailwinds in the US and EU are accelerating telehealth adoption.


In [9]:
# Demo 8 - How LLMs Support Reasoning and Agent Behavior

client = openai.OpenAI()

# --- Capability 1: Task Decomposition (MECE Breakdown) ---
print("CAPABILITY 1: Task Decomposition (MECE Breakdown)")
print("=" * 60)

decomposition_response = client.chat.completions.create(
    model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
    messages=[
        {"role": "system", "content": "You are a McKinsey engagement manager. Break down client problems into MECE (Mutually Exclusive, Collectively Exhaustive) workstreams with numbered sub-tasks."},
        {"role": "user", "content": "Our pharmaceutical client wants to evaluate entering the biosimilars market in Europe. Structure the key workstreams for this engagement."}
    ],
    temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0.3")),
    max_tokens=int(os.getenv("DEFAULT_MAX_TOKENS", "300"))
)
print(decomposition_response.choices[0].message.content)

# --- Capability 2: Planning (Engagement Plan) ---
print("\n\nCAPABILITY 2: Planning — Generating an Engagement Plan")
print("=" * 60)

planning_response = client.chat.completions.create(
    model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
    messages=[
        {"role": "system", "content": (
            "You are a McKinsey project manager. Given an engagement goal, output a numbered "
            "week-by-week plan of concrete actions. Each step should be a single deliverable. "
            "Format: 'Week N: [DELIVERABLE] — description'"
        )},
        {"role": "user", "content": "Goal: Execute a 6-week due diligence for a private equity client evaluating a $500M acquisition of a B2B SaaS company."}
    ],
    temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0.3")),
    max_tokens=int(os.getenv("DEFAULT_MAX_TOKENS", "300"))
)
print(planning_response.choices[0].message.content)

# --- Capability 3: Self-Reflection (Quality Review) ---
print("\n\nCAPABILITY 3: Self-Reflection — Reviewing and Improving Analysis")
print("=" * 60)

# First, generate a deliberately brief answer
initial_answer = client.chat.completions.create(
    model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
    messages=[{"role": "user", "content": "What are the key drivers of customer churn in the telecom industry?"}],
    max_tokens=int(os.getenv("DEFAULT_MAX_TOKENS", "40"))
)
brief = initial_answer.choices[0].message.content
print(f"Initial (brief) answer: {brief}")

# Now ask the model to reflect on and improve its own answer — like a partner review
reflection = client.chat.completions.create(
    model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
    messages=[
        {"role": "system", "content": "You are a McKinsey Senior Partner reviewing an associate's work. Evaluate the answer below for completeness, MECE structure, and analytical rigor, then provide an improved version."},
        {"role": "user", "content": f"Original question: What are the key drivers of customer churn in the telecom industry?\n\nAssociate's draft: {brief}\n\nProvide your critique and an improved, partner-ready answer."}
    ],
    max_tokens=int(os.getenv("DEFAULT_MAX_TOKENS", "250"))
)
print(f"\nPartner review & improvement:\n{reflection.choices[0].message.content}")

# --- Capability 4: Tool Selection (Consulting Tools) ---
print("\n\nCAPABILITY 4: Tool Selection — Choosing the Right Analytical Tool")
print("=" * 60)

tool_prompt = """You have access to these McKinsey analytical tools:
1. market_database — for market size, growth rates, and competitive landscape data
2. financial_model — for DCF, LBO, and valuation calculations
3. survey_platform — for customer/employee surveys and NPS analysis
4. expert_network — for scheduling calls with industry experts
5. benchmarking_tool — for comparing client KPIs against industry benchmarks

For each request, respond with the tool you would select and why. Format: TOOL: [name] | REASON: [explanation]"""

tool_selection = client.chat.completions.create(
    model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
    messages=[
        {"role": "system", "content": tool_prompt},
        {"role": "user", "content": "What is the total addressable market for electric vehicle charging infrastructure in North America?"}
    ],
    max_tokens=int(os.getenv("DEFAULT_MAX_TOKENS", "100"))
)
print(f"Request: 'TAM for EV charging infrastructure in North America'")
print(f"Decision: {tool_selection.choices[0].message.content}")

tool_selection2 = client.chat.completions.create(
    model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
    messages=[
        {"role": "system", "content": tool_prompt},
        {"role": "user", "content": "Is the target company's EBITDA margin competitive relative to peers in the specialty chemicals sector?"}
    ],
    max_tokens=int(os.getenv("DEFAULT_MAX_TOKENS", "100"))
)
print(f"\nRequest: 'Is target EBITDA margin competitive vs specialty chemicals peers?'")
print(f"Decision: {tool_selection2.choices[0].message.content}")

CAPABILITY 1: Task Decomposition (MECE Breakdown)
To evaluate the potential entry of our pharmaceutical client into the biosimilars market in Europe, we can break down the engagement into the following MECE workstreams:

### Workstream 1: Market Analysis
1. **Market Size and Growth Potential**
   - 1.1: Analyze current market size of biosimilars in Europe.
   - 1.2: Assess historical growth rates and future projections.
   - 1.3: Identify key drivers of market growth (e.g., regulatory changes, healthcare cost pressures).

2. **Competitive Landscape**
   - 2.1: Identify key competitors in the European biosimilars market.
   - 2.2: Analyze competitor product portfolios and market shares.
   - 2.3: Evaluate competitor pricing strategies and market positioning.

3. **Regulatory Environment**
   - 3.1: Review European Medicines Agency (EMA) guidelines for biosimilars.
   - 3.2: Assess the approval process and timelines for biosimilars in Europe.
   - 3.3: Identify potential regulatory hurdl

### Task 3: Experiment with Model Parameters

**Approach:** We loop over three temperature values (0.0, 0.5, 1.0), sending the exact same prompt and model each time. Results are stored in a dictionary keyed by temperature. Using `temperature=0.0` should give nearly identical outputs on repeated runs, while `temperature=1.0` will produce more varied and creative responses.

> ---
> **INSTRUCTOR GUIDANCE**
>
> **How to Conduct This Exercise:** Give students 5-7 minutes. This is a straightforward loop exercise. Encourage students to try different prompts beyond the provided test to see how temperature affects creative vs. analytical outputs. After the exercise, ask: 'When would you use temperature=0 in a consulting context?' (Answer: financial calculations, data extraction, classification) vs. 'When would you use temperature=1.0?' (Answer: brainstorming, creative strategy ideation).
>
> **Common Student Mistakes:**
> - Setting both `temperature` and `top_p` at the same time -- OpenAI recommends changing only one
> - Using a dictionary with float keys and then trying to access with `results[0]` instead of `results[0.0]`
> - Creating a new `openai.OpenAI()` client inside the loop (wasteful but not incorrect -- discuss efficiency)
> - Not returning the results dictionary -- just printing inside the function
>
> **Skippable?** YES -- CAN SKIP if running behind schedule. The concept is already well-demonstrated in Demo 3. If skipping, briefly show the solution and move directly to Task 4 which is more important for Day 2 foundations.

In [10]:
# Task 3 - SOLUTION: Experiment with Model Parameters

def compare_temperatures(prompt):
    """Send the same prompt at different temperatures and return all responses.

    Returns a dictionary: {temperature_value: response_text}
    """
    # Step 1: Initialize the client.
    client = openai.OpenAI()

    # Step 2: Define the temperatures to compare.
    temperatures = [0.0, 0.5, 1.0]
    results = {}

    # Step 3: Loop over temperatures, making one API call per setting.
    for temp in temperatures:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            temperature=temp,
            max_tokens=150
        )
        results[temp] = response.choices[0].message.content

    return results


# Test with a McKinsey-relevant prompt
test_prompt = "Describe the competitive advantages of a vertically integrated supply chain for a consumer goods company."
results = compare_temperatures(test_prompt)

print(f"Prompt: {test_prompt}")
print("=" * 60)
for temp, text in results.items():
    print(f"\nTemperature {temp}:")
    print(f"  {text}")
    print()

Prompt: Describe the competitive advantages of a vertically integrated supply chain for a consumer goods company.

Temperature 0.0:
  A vertically integrated supply chain offers several competitive advantages for a consumer goods company. Here are some key benefits:

1. **Cost Control**: By owning multiple stages of the supply chain, from raw materials to manufacturing to distribution, a company can reduce costs associated with outsourcing and third-party suppliers. This can lead to better pricing strategies and improved profit margins.

2. **Quality Assurance**: Vertical integration allows a company to maintain greater control over the quality of its products. By managing the entire production process, from sourcing materials to final delivery, a company can ensure that its standards are met consistently.

3. **Supply Chain Efficiency**: A vertically integrated supply chain can streamline operations, reduce lead times, and improve overall efficiency. Companies can better coordinate pr

### Task 4: Build a Simple Conversational Agent Foundation

**Approach:** The `SimpleAgent` class simulates a McKinsey engagement manager assistant. It maintains an internal `messages` list starting with a system message defining a consulting persona. Each call to `chat()` appends the user message, sends the full history (so the model remembers prior context — like remembering the client industry across turns), and appends the assistant's response. This is the foundation for multi-turn consulting assistants.

> ---
> **INSTRUCTOR GUIDANCE**
>
> **How to Conduct This Exercise:** Give students 10-12 minutes -- this is the most important task of Session 1. Suggest building incrementally: first get `__init__` working with the system message, then `chat()` with a single turn, then verify multi-turn context retention. The key 'aha moment' is when the second question references context from the first turn and the model remembers. Walk through the message history at the end to show students exactly what the model receives. This pattern is the foundation for everything in Day 2.
>
> **Common Student Mistakes:**
> - Not including the system message in the initial `self.messages` list -- the persona never takes effect
> - Appending only the user message but forgetting to append the assistant's response -- context is lost after each turn
> - Sending only the latest message instead of the full `self.messages` history -- destroys multi-turn memory
> - Initializing `self.client = openai.OpenAI()` outside `__init__` at the class level (works but is fragile)
> - Returning `response` instead of `response.choices[0].message.content` from the `chat()` method
>
> **Skippable?** NO -- CRITICAL -- this introduces the conversational agent pattern (system message + history management) that is the foundation for every agent built on Days 2 and 3. Do NOT skip.

In [11]:
# Task 4 - SOLUTION: Build a Simple Conversational Agent Foundation

class SimpleAgent:
    def __init__(self):
        """Initialize the agent with a McKinsey consulting persona."""
        # Step 1: Create the OpenAI client.
        self.client = openai.OpenAI()

        # Step 2: Define the system message as a McKinsey engagement manager.
        self.system_message = (
            "You are a McKinsey engagement manager assistant. You help consultants with "
            "client engagement planning, market analysis, and structured problem-solving. "
            "Use MECE frameworks, provide data-driven recommendations, and maintain a "
            "professional consulting tone. When asked a question, first structure the problem, "
            "then provide actionable insights."
        )

        # Step 3: Initialize the message history with the system message.
        self.messages = [
            {"role": "system", "content": self.system_message}
        ]

    def chat(self, user_input):
        """Process a user message and return the assistant's response."""
        # Step 1: Append the user's message to the conversation history.
        self.messages.append({"role": "user", "content": user_input})

        # Step 2: Send the FULL history to the LLM.
        response = self.client.chat.completions.create(
            model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
            messages=self.messages,
            temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0.7")),
            max_tokens=int(os.getenv("DEFAULT_MAX_TOKENS", "500"))
        )

        # Step 3: Extract the assistant's reply.
        assistant_reply = response.choices[0].message.content

        # Step 4: Append the assistant's reply to the history for future context.
        self.messages.append({"role": "assistant", "content": assistant_reply})

        return assistant_reply

    def get_history(self):
        """Return the full conversation history."""
        return self.messages


# Test the agent with a multi-turn McKinsey engagement conversation
agent = SimpleAgent()

# Turn 1
print("USER: We have a new client in the automotive sector facing margin pressure. What should we focus on?")
print(f"AGENT: {agent.chat('We have a new client in the automotive sector facing margin pressure. What should we focus on?')}")
print()

# Turn 2 -- the agent should remember the client context (automotive)
print("USER: Which of those areas would have the quickest impact?")
print(f"AGENT: {agent.chat('Which of those areas would have the quickest impact?')}")
print()

# Show conversation history length
history = agent.get_history()
print(f"Conversation history: {len(history)} messages")
for msg in history:
    print(f"  [{msg['role']}] {msg['content'][:80]}{'...' if len(msg['content']) > 80 else ''}")

USER: We have a new client in the automotive sector facing margin pressure. What should we focus on?
AGENT: To address the margin pressure faced by your client in the automotive sector, we can structure the problem using a MECE framework. This will help us identify key areas of focus and actionable insights.

### Problem Structure

1. **Cost Structure Analysis**
   - **Fixed Costs**: Evaluate overhead, R&D, and manufacturing costs.
   - **Variable Costs**: Analyze raw materials, labor, and logistics expenses.

2. **Revenue Streams**
   - **Product Mix**: Assess the profitability of different vehicle models and segments (e.g., electric vs. combustion engines).
   - **Sales Channels**: Examine the effectiveness of direct sales, dealerships, and online platforms.

3. **Market Dynamics**
   - **Competitive Landscape**: Identify key competitors and their pricing strategies.
   - **Consumer Trends**: Understand shifts in consumer preferences, such as the move towards electric vehicles (EVs) 

### Task 5: Generate and Compare Embeddings for Semantic Search

**Approach:** We use the OpenAI Embeddings API to convert consulting phrases into dense vectors, then compute a cosine similarity matrix using NumPy. This reveals which phrases are semantically related — the foundation for building knowledge-base search and RAG systems over McKinsey engagement documents.

> ---
> **INSTRUCTOR GUIDANCE**
>
> **How to Conduct:** Start by explaining embeddings visually - 'words as coordinates in space.' Have students generate embeddings for similar and dissimilar sentences, then compute cosine similarity. Compare results as a class to build intuition for semantic search.
>
> **Common Student Mistakes:**
> - Confusing token count with embedding dimensionality
> - Not normalizing embeddings before computing cosine similarity
> - Using very short single-word inputs that produce poor embeddings
> - Forgetting that embeddings are model-specific and not interchangeable
>
> **Skippable?** YES
> Embeddings are covered more thoroughly in Day 3 RAG sessions. If time is tight, demonstrate the instructor solution and briefly explain the concept. Students will get hands-on embedding practice during RAG.
> ---


In [12]:
# Task 5 - SOLUTION: Generate and Compare Embeddings for Semantic Search

import numpy as np

def get_embeddings(texts):
    """Generate embeddings for a list of texts using the OpenAI API.

    Returns a dict mapping each text to its embedding vector.
    """
    client = openai.OpenAI()

    # Step 1: Call the embeddings API with all texts at once (batch is more efficient).
    response = client.embeddings.create(
        model=os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small"),
        input=texts
    )

    # Step 2: Extract the embedding vectors.
    embeddings = {text: item.embedding for text, item in zip(texts, response.data)}
    return embeddings


def cosine_similarity(vec_a, vec_b):
    """Compute cosine similarity between two vectors."""
    a = np.array(vec_a)
    b = np.array(vec_b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))


# Step 3: Define McKinsey consulting phrases to compare.
consulting_phrases = [
    "Revenue growth decelerated in Q3 due to market headwinds",
    "Sales momentum slowed in the third quarter",
    "The company needs to restructure its operating model",
    "Post-merger integration is behind schedule",
    "Customer acquisition costs are rising faster than lifetime value",
    "The cost of acquiring new customers exceeds their long-term profitability",
]

embeddings = get_embeddings(consulting_phrases)
print(f"Generated embeddings for {len(embeddings)} phrases")
print(f"Embedding dimension: {len(list(embeddings.values())[0])}")

# Step 4: Build a similarity matrix.
print("\nCosine Similarity Matrix:")
print("=" * 60)
for i, text_a in enumerate(consulting_phrases):
    for j, text_b in enumerate(consulting_phrases):
        if j > i:
            sim = cosine_similarity(embeddings[text_a], embeddings[text_b])
            if sim > 0.7:  # Only show high-similarity pairs
                print(f"\nSimilarity: {sim:.3f}")
                print(f"  A: {text_a}")
                print(f"  B: {text_b}")

Generated embeddings for 6 phrases
Embedding dimension: 1536

Cosine Similarity Matrix:

Similarity: 0.721
  A: Revenue growth decelerated in Q3 due to market headwinds
  B: Sales momentum slowed in the third quarter


### Task 6: Build a Multi-Step Analysis Pipeline with Quality Review

**Approach:** We chain three LLM calls in sequence — (1) analyze a client problem, (2) generate strategic recommendations, (3) have a "Senior Partner" persona critically review the recommendations. The third step implements a self-reflection pattern where the LLM evaluates its own prior output, a key capability for reliable agentic systems.

> ---
> **INSTRUCTOR GUIDANCE**
>
> **How to Conduct:** Walk through the pipeline design first - what are the stages and how does output flow between them? Have students build one stage at a time, testing each before connecting them. Emphasize the quality review step as the 'McKinsey quality bar.'
>
> **Common Student Mistakes:**
> - Not passing structured output from one stage to the next properly
> - Skipping the quality review stage or making it a no-op
> - Using overly long prompts that exceed context when chained
> - Not handling cases where intermediate stages produce unexpected output
>
> **Skippable?** YES
> Multi-step pipelines are covered more deeply in Day 2 with LangChain. Show the instructor solution and discuss the concept. Students will build more sophisticated pipelines tomorrow.
> ---


In [13]:
# Task 6 - SOLUTION: Build a Multi-Step Analysis Pipeline with Quality Review

client = openai.OpenAI()

def analysis_pipeline(client_problem):
    """Run a 3-step consulting analysis pipeline with quality review.

    Step 1: Analyze the problem (Associate level)
    Step 2: Generate recommendations (Engagement Manager level)
    Step 3: Quality review (Senior Partner level)

    Returns a dict with outputs from each step.
    """
    # Step 1: Problem Analysis
    analysis = client.chat.completions.create(
        model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
        messages=[
            {"role": "system", "content": "You are a McKinsey Associate. Analyze the client problem using a MECE framework. Identify 3-4 key issue areas and the data you would need to investigate each."},
            {"role": "user", "content": client_problem}
        ],
        temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0.3")),
        max_tokens=int(os.getenv("DEFAULT_MAX_TOKENS", "400"))
    ).choices[0].message.content

    # Step 2: Generate Recommendations
    recommendations = client.chat.completions.create(
        model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
        messages=[
            {"role": "system", "content": "You are a McKinsey Engagement Manager. Based on the analysis below, generate 3 actionable recommendations. Each should have: (1) the recommendation, (2) expected impact, (3) implementation timeline."},
            {"role": "user", "content": f"Analysis:\n{analysis}\n\nGenerate strategic recommendations."}
        ],
        temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0.5")),
        max_tokens=int(os.getenv("DEFAULT_MAX_TOKENS", "400"))
    ).choices[0].message.content

    # Step 3: Quality Review (Self-Reflection)
    review = client.chat.completions.create(
        model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
        messages=[
            {"role": "system", "content": "You are a McKinsey Senior Partner reviewing work before client delivery. Evaluate the analysis and recommendations for: (1) MECE completeness, (2) analytical rigor, (3) actionability. Provide a rating (Strong/Adequate/Needs Work) and specific improvement suggestions."},
            {"role": "user", "content": f"Client Problem: {client_problem}\n\nAnalysis:\n{analysis}\n\nRecommendations:\n{recommendations}"}
        ],
        temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0.3")),
        max_tokens=int(os.getenv("DEFAULT_MAX_TOKENS", "300"))
    ).choices[0].message.content

    return {"analysis": analysis, "recommendations": recommendations, "review": review}


# Test the pipeline
problem = "A mid-size European bank is losing market share in retail deposits to digital-first neobanks. Their NPS has dropped 20 points in 18 months."

result = analysis_pipeline(problem)

print("STEP 1: Problem Analysis (Associate)")
print("=" * 60)
print(result["analysis"])

print("\nSTEP 2: Strategic Recommendations (EM)")
print("=" * 60)
print(result["recommendations"])

print("\nSTEP 3: Quality Review (Senior Partner)")
print("=" * 60)
print(result["review"])

STEP 1: Problem Analysis (Associate)
To analyze the problem of the mid-size European bank losing market share in retail deposits to digital-first neobanks and experiencing a significant drop in Net Promoter Score (NPS), we can use the MECE (Mutually Exclusive, Collectively Exhaustive) framework. This approach will help us identify key issue areas and the data needed to investigate each.

### Key Issue Areas

1. **Customer Experience and Satisfaction**
   - **Data Needed:**
     - Detailed NPS survey results segmented by customer demographics (age, income, etc.)
     - Customer feedback and complaints analysis (qualitative data)
     - Comparison of customer service metrics (response times, resolution rates) with neobanks
     - User experience (UX) assessments of the bank's digital platforms (website, mobile app)

2. **Product Offering and Competitiveness**
   - **Data Needed:**
     - Analysis of the bank's current retail deposit products (interest rates, fees, features) compared to n

### Task 7: Estimate API Costs from Token Usage

**Approach:** We combine `tiktoken` for counting input tokens with the API response's `usage` field for actual input/output counts. We then apply OpenAI's per-token pricing to estimate the cost. This is essential for engagement budgeting — a partner needs to know what an AI-augmented engagement will cost before staffing the team.

> ---
> **INSTRUCTOR GUIDANCE**
>
> **How to Conduct:** Frame this as a real business concern - 'How much does this cost per engagement?' Have students extract token counts from API responses and calculate costs using published pricing. Compare costs across different prompt strategies from earlier tasks.
>
> **Common Student Mistakes:**
> - Using wrong pricing tiers (confusing input vs output token prices)
> - Not accounting for both prompt and completion tokens
> - Hardcoding prices instead of making them configurable
> - Forgetting that different models have different pricing
>
> **Skippable?** YES
> Cost estimation is useful but not essential for the technical flow. If behind schedule, briefly show the calculation and share the pricing reference. Students can do this independently.
> ---


In [14]:
# Task 7 - SOLUTION: Estimate API Costs from Token Usage

def estimate_cost(prompt, system_message="You are a helpful assistant.", model=None):
    """Send a prompt and return the response along with token usage and estimated cost.

    Returns a dict with: response, input_tokens, output_tokens, total_tokens, estimated_cost_usd
    """
    if model is None:
        model = os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini")

    # Step 1: Pricing per 1M tokens (as of 2024 for gpt-4o-mini)
    pricing = {
        "gpt-4o-mini": {"input": 0.15, "output": 0.60},
        "gpt-4o": {"input": 2.50, "output": 10.00},
    }
    model_pricing = pricing.get(model, pricing["gpt-4o-mini"])

    # Step 2: Make the API call.
    client = openai.OpenAI()
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_message},
            {"role": "user", "content": prompt}
        ],
        temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0")),
        max_tokens=int(os.getenv("DEFAULT_MAX_TOKENS", "200"))
    )

    # Step 3: Extract token usage from the response.
    usage = response.usage
    input_tokens = usage.prompt_tokens
    output_tokens = usage.completion_tokens

    # Step 4: Calculate cost.
    cost = (input_tokens * model_pricing["input"] / 1_000_000) + \
           (output_tokens * model_pricing["output"] / 1_000_000)

    return {
        "response": response.choices[0].message.content,
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "total_tokens": input_tokens + output_tokens,
        "estimated_cost_usd": cost
    }


# Test with a consulting prompt
result = estimate_cost(
    prompt="Outline the key steps for conducting a commercial due diligence for a PE acquisition target in the SaaS industry.",
    system_message="You are a McKinsey engagement manager specializing in M&A due diligence."
)

print(f"Response: {result['response'][:200]}...")
print(f"\nToken Usage:")
print(f"  Input tokens : {result['input_tokens']}")
print(f"  Output tokens: {result['output_tokens']}")
print(f"  Total tokens : {result['total_tokens']}")
print(f"  Estimated cost: ${result['estimated_cost_usd']:.6f}")

# Estimate cost for a full engagement (e.g., 500 similar queries)
engagement_queries = 500
total_cost = result['estimated_cost_usd'] * engagement_queries
print(f"\nProjected cost for {engagement_queries} queries: ${total_cost:.4f}")

Response: Conducting commercial due diligence for a private equity (PE) acquisition target in the Software as a Service (SaaS) industry involves a systematic approach to assess the target's market position, gro...

Token Usage:
  Input tokens : 47
  Output tokens: 300
  Total tokens : 347
  Estimated cost: $0.000187

Projected cost for 500 queries: $0.0935


### Task 8: Build a Context-Aware Routing Agent

**Approach:** We build a two-stage agent: first, a "router" LLM call classifies the user's question into a McKinsey practice area (Strategy, Operations, Digital, or M&A). Then, a second LLM call uses a practice-specific system message to generate an expert response. This pattern — classify then route — is fundamental to multi-agent architectures you will build in Day 2.

> ---
> **INSTRUCTOR GUIDANCE**
>
> **How to Conduct:** This is the capstone task for Session 1. Explain routing as a pattern: 'different queries need different handling.' Have students define 2-3 routes, build the router logic, then test with varied queries. Discuss how this maps to real McKinsey AI assistant architecture.
>
> **Common Student Mistakes:**
> - Making the routing logic too rigid with exact string matching instead of semantic understanding
> - Not handling the fallback case when no route matches
> - Building overly complex routing when simple if/else would suffice for the demo
> - Forgetting to include context from previous steps in the routed prompts
>
> **Skippable?** YES
> Routing agents are covered extensively in Day 2 LangGraph session. If running long, walk through the solution conceptually and note they'll build more sophisticated agents tomorrow.
> ---


In [15]:
# Task 8 - SOLUTION: Build a Context-Aware Routing Agent

client = openai.OpenAI()

# Step 1: Define practice-specific system messages.
PRACTICE_PERSONAS = {
    "strategy": "You are a McKinsey Senior Partner in Strategy & Corporate Finance. Focus on competitive positioning, portfolio strategy, market entry, and value creation. Use frameworks like Porter's Five Forces and growth-share matrices.",
    "operations": "You are a McKinsey Operations expert. Focus on supply chain optimization, lean manufacturing, procurement savings, and operational KPIs. Be quantitative and suggest specific analyses.",
    "digital": "You are a McKinsey Digital & Analytics leader. Focus on digital transformation, data strategy, AI/ML use cases, and technology architecture. Recommend modern tech solutions.",
    "m&a": "You are a McKinsey M&A and Due Diligence specialist. Focus on deal structuring, synergy estimation, integration planning, and target valuation. Be precise about financial metrics."
}


def route_and_respond(question):
    """Classify the question into a practice area, then respond with a specialist persona.

    Returns a dict with: practice_area, expert_response
    """
    # Step 2: Route — classify the question.
    router_response = client.chat.completions.create(
        model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
        messages=[
            {"role": "system", "content": "Classify the following consulting question into exactly one practice area. Respond with ONLY one word: strategy, operations, digital, or m&a"},
            {"role": "user", "content": question}
        ],
        temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0")),
        max_tokens=int(os.getenv("DEFAULT_MAX_TOKENS", "5"))
    )
    practice = router_response.choices[0].message.content.strip().lower()

    # Step 3: Get the matching persona (default to strategy if unrecognized).
    persona = PRACTICE_PERSONAS.get(practice, PRACTICE_PERSONAS["strategy"])

    # Step 4: Respond with the specialist persona.
    expert_response = client.chat.completions.create(
        model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
        messages=[
            {"role": "system", "content": persona},
            {"role": "user", "content": question}
        ],
        temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0.5")),
        max_tokens=int(os.getenv("DEFAULT_MAX_TOKENS", "300"))
    )

    return {
        "practice_area": practice,
        "expert_response": expert_response.choices[0].message.content
    }


# Test with different types of consulting questions
questions = [
    "How should we evaluate entering the Indian electric vehicle market?",
    "Our client's warehouse fulfillment time is 40% above industry benchmark. How do we fix this?",
    "What AI/ML use cases should a retail bank prioritize for 2025?",
    "We're acquiring a competitor for $2B. How should we estimate cost synergies?"
]

for q in questions:
    result = route_and_respond(q)
    print(f"\nQuestion: {q}")
    print(f"Routed to: {result['practice_area'].upper()}")
    print(f"Response: {result['expert_response'][:200]}...")
    print("-" * 60)


Question: How should we evaluate entering the Indian electric vehicle market?
Routed to: STRATEGY
Response: Entering the Indian electric vehicle (EV) market presents a significant opportunity, but it requires a thorough evaluation of various factors to ensure a successful market entry strategy. Here’s a str...
------------------------------------------------------------

Question: Our client's warehouse fulfillment time is 40% above industry benchmark. How do we fix this?
Routed to: OPERATIONS
Response: To address the issue of your client's warehouse fulfillment time being 40% above the industry benchmark, we can take a structured approach focusing on supply chain optimization, lean manufacturing pri...
------------------------------------------------------------

Question: What AI/ML use cases should a retail bank prioritize for 2025?
Routed to: DIGITAL
Response: For a retail bank looking to prioritize AI/ML use cases for 2025, the following areas should be considered, as they align 

---
## Summary

In this session students learned the foundational skills for working with LLMs programmatically:

1. **API Connections** -- How to initialize the OpenAI client and make chat completion requests.
2. **Tokenization** -- How text is converted to tokens, why token counts matter for cost and context window limits, and how to use `tiktoken` to measure them.
3. **Model Parameters** -- How `temperature`, `top_p`, and `max_tokens` influence the style, creativity, and length of LLM outputs.
4. **System Messages & Personas** -- How the system message acts as a powerful control lever to change LLM behavior without changing the user prompt.
5. **LLM Pipelines** -- How to chain multiple LLM calls together so the output of one step feeds into the next, forming the basis for agentic workflows.
6. **Embeddings & Vector Representations** -- How to generate embeddings using the OpenAI API and compute cosine similarity to measure semantic relatedness between texts.
7. **Embedding Visualization** -- How to use PCA and t-SNE to project high-dimensional embeddings into 2D for visual inspection, and how to build similarity heatmaps.
8. **LLM Reasoning & Agent Behavior** -- How LLMs exhibit emergent capabilities (task decomposition, planning, self-reflection, tool selection) that make them suitable as the core of agentic systems.

**Instructor Notes:**
- Encourage students to experiment with different prompts in the demos before moving to the tasks.
- For Task 2, discuss edge cases: what if the prompt has no periods? What about very long single sentences?
- For Task 4, discuss how the message history grows and what happens when it exceeds the context window.
- For Demos 6-7, highlight how embedding similarity is the foundation for RAG (Retrieval-Augmented Generation).
- For Demo 8, discuss the difference between "narrow" tool-use and "general" reasoning — both are needed for robust agents.

**Up Next:** In Session 2, we will dive into advanced prompting techniques -- including few-shot prompting, chain-of-thought reasoning, and structured output generation -- that make LLMs more reliable and useful as the reasoning core of agentic systems.